# Dokumentacja Projektu: Super-Resolution & Denoising

Niniejszy notatnik prezentuje dokumentację, metodologię oraz wyniki eksperymentów dla projektu z obszaru przetwarzania obrazów (Computer Vision). Celem projektu jest badanie architektur głębokich sieci neuronowych w dwóch klasycznych zadaniach:
1. **Denoising (Odszumianie)** – usuwanie sztucznie dodanego szumu z obrazów.
2. **Super-Resolution (Zwiększanie rozdzielczości)** – rekonstrukcja obrazów wysokiej jakości z rozmytych odpowiedników o niskiej rozdzielczości.

Wszystkie eksperymenty były realizowane przy użyciu biblioteki PyTorch na zbiorze obrazów **DIV2K**.

## 1. Zbiór danych i Przetwarzanie (Data Pipeline)

Do treningu i walidacji wykorzystano wysokorozdzielczy zbiór **DIV2K**. Przygotowanie danych (zaimplementowane w `datasets.py`) różni się w zależności od zadania:

### Denoising (Odszumianie)
* **Obraz docelowy (Target):** Wykadrowany i przeskalowany do stałego rozmiaru `256x256` przy użyciu interpolacji bikubicznej (`INTER_CUBIC`). Znormalizowany do przedziału `[0, 1]`.
* **Obraz wejściowy (Input):** Do obrazu docelowego dodawany jest losowy szum Gaussa (o wariancji zależnej od wartości `sigma` losowanej z przedziału 0.01-0.03).

### Super-Resolution (Zwiększanie rozdzielczości)
* **Obraz docelowy (HR - High Resolution):** Podobnie jak wyżej, przeskalowany do rozmiaru `256x256` (`INTER_CUBIC`).
* **Obraz wejściowy (LR - Low Resolution) - WPROWADZENIE DEGRADACJI:** W celu zasymulowania obrazu o niskiej jakości, obraz oryginalny jest najpierw pomniejszany do rozmiaru `32x32` lub `64x64` (wybierane losowo) za pomocą algorytmu **`INTER_AREA`** (który dobrze radzi sobie z zachowaniem informacji przy pomniejszaniu). 
  Następnie, aby wymiary pasowały do wejścia sieci, obraz jest powiększany z powrotem do rozmiaru wejściowego `256x256` przy użyciu **interpolacji bikubicznej (`INTER_CUBIC`)**. Powoduje to silne rozmycie i utratę detali (efekt "pikselozy" i "bluru"), który sieć musi nauczyć się odwracać.

## 2. Architektury Modeli i Hiperparametry

W projekcie zaimplementowano, przetestowano i porównano trzy różne struktury sieci konwolucyjnych (zdefiniowane w `models.py`):

1. **SimpleUNet:** Podstawowa wersja architektury U-Net, składająca się z jednej warstwy kodera, warstwy łączącej (MaxPool) i prostego dekodera. Jest to szybki i lekki model.
2. **BetterUNet:** Rozbudowana wersja U-Net. Wprowadza autorskie bloki `DoubleConv` (dwie konwolucje przeplatane normalizacją `BatchNorm2d` i aktywacją `ReLU`). Posiada głębszą strukturę kodera/dekodera oraz wykorzystuje połączenia omijające (skip-connections), które łączą cechy z kodera bezpośrednio do dekodera, zapobiegając utracie detali przestrzennych.
3. **ResNetRestoration:** Architektura bazująca na blokach rezydualnych (`ResidualBlock`). Wykorzystuje globalne połączenie omijające – wejście sieci jest dodawane bezpośrednio do wyjścia bloku konwolucyjnego przed finalną aktywacją Sigmoid. Sprawia to, że sieć uczy się jedynie "różnicy" (rezyduum) pomiędzy obrazem zepsutym a naprawionym, co jest bardzo wydajne w zadaniach restauracji obrazu.

### Funkcje straty (Loss Functions)
Podczas treningów testowano wpływ różnych funkcji optymalizacji na jakość rekonstrukcji obrazu:
* **MSELoss (L2):** Standardowy błąd średniokwadratowy.
* **L1Loss:** Błąd bezwzględny (często daje ostrzejsze obrazy niż MSE).
* **CombinedPerceptualLoss:** Własna implementacja złożonej funkcji straty, która optymalizuje parametry łącząc błąd L1 (waga 1.0) z metryką perceptyjną LPIPS wykorzystującą sieć **VGG** (waga 0.1). Skupia się ona na optymalizacji wizualnych struktur, które są zauważalne dla ludzkiego oka.

## 3. Metryki Ewaluacyjne i Wyniki Eksperymentów

Każdy z wytrenowanych modeli został oceniony na zbiorze walidacyjnym za pomocą trzech kluczowych metryk (funkcja `calculate_metrics` w `metrics.py`):
* **PSNR (Peak Signal-to-Noise Ratio):** Mierzy obiektywną jakość obrazu na podstawie błędu na poziomie pikseli (wyższa wartość jest lepsza).
* **SSIM (Structural Similarity Index):** Skupia się na podobieństwie struktur widocznych na obrazie, jasności i kontraście (wartość bliższa 1.0 jest lepsza).
* **LPIPS (Learned Perceptual Image Patch Similarity):** Wykorzystuje wstępnie wytrenowaną sieć VGG do oceny odległości percepcyjnej między obrazami. Lepiej oddaje to, jak jakość obrazu postrzega ludzkie oko (niższa wartość jest lepsza).

Poniższy kod przeszukuje folder `outputs`, wybiera najlepsze checkpointy (.pth) i przeprowadza ewaluację na zbiorze testowym. Wyniki są zapisywane do pliku `all_metrics.csv` i wyświetlane w formie tabeli.

In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../classes'))

from evaluate_model import evaluate_all_models

evaluate_all_models()

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: d:\studia\Studia 2 stopnia\Semestr 3\SIGK\SIGK-projects\.venv\Lib\site-packages\lpips\weights\v0.1\vgg.pth
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: d:\studia\Studia 2 stopnia\Semestr 3\SIGK\SIGK-projects\.venv\Lib\site-packages\lpips\weights\v0.1\vgg.pth

Evaluating best model for super_resolution | Criterion: CombinedPerceptualLoss | LR: 0.0001 | Model: BetterUNet | Epoch: 20
Ocenianie na: cuda
Rozpoczynam ewaluację 100 obrazów...

Evaluating best model for super_resolution | Criterion: L1Loss | LR: 0.0001 | Model: BetterUNet | Epoch: 20
Ocenianie na: cuda
Rozpoczynam ewaluację 100 obrazów...

Evaluating best model for super_resolution | Criterion: L1Loss | LR: 0.0005 | Model: BetterUNet | Epoch: 20
Ocenianie na: cuda
Rozpoczynam ewaluację 100 obrazów...

Evaluating best model for super_resolution | Criterion: MSELoss | LR: 0.0001 | Model: Better

Wyniki pobrane z pliku csv `all_metrics.csv` w formie tabeli:

In [2]:
import pandas as pd

csv_file = "../outputs/eval_results/all_metrics.csv"

df_results = pd.read_csv(csv_file)

df_sorted = df_results.sort_values(by="PSNR", ascending=False)

display(df_sorted)

best_model_info = df_sorted.iloc[0]
print(f"Best model: {best_model_info['Model File']}")
print(f"Trained with Criterion: {best_model_info['Criterion']} and LR: {best_model_info['Learning Rate']}")

,Task,Criterion,Learning Rate,Model File,Samples,PSNR,SSIM,LPIPS
7,super_resolution,L1Loss,0.0005,ResNetRestoration_model_epoch_20_super_resolut...,100,19.608329,0.503481,0.533365
0,super_resolution,CombinedPerceptualLoss,0.0001,BetterUNet_model_epoch_20_super_resolution_0.1...,100,19.572103,0.507953,0.408026
9,super_resolution,MSELoss,0.0005,ResNetRestoration_model_epoch_20_super_resolut...,100,19.531339,0.498062,0.532824
2,super_resolution,L1Loss,0.0005,BetterUNet_model_epoch_20_super_resolution_0.0...,100,19.493334,0.500538,0.531630
8,super_resolution,MSELoss,0.0001,ResNetRestoration_model_epoch_20_super_resolut...,100,19.452851,0.491374,0.534597
3,super_resolution,MSELoss,0.0001,BetterUNet_model_epoch_20_super_resolution_0.0...,100,19.433840,0.487794,0.528986
12,super_resolution,L1Loss,0.0005,SimpleUNet_model_epoch_20_super_resolution_0.0...,100,19.418272,0.496917,0.519769
6,super_resolution,L1Loss,0.0001,ResNetRestoration_model_epoch_20_super_resolut...,100,19.356983,0.486303,0.543275
1,super_resolution,L1Loss,0.0001,BetterUNet_model_epoch_20_super_resolution_0.0...,100,19.347359,0.489079,0.533489
5,super_resolution,CombinedPerceptualLoss,0.0001,ResNetRestoration_model_epoch_20_super_resolut...,100,19.311995,0.491358,0.513040


Best model: ResNetRestoration_model_epoch_20_super_resolution_0.0707.pth
Trained with Criterion: L1Loss and LR: 0.0005


## 4. Ewaluacja Rozwiązań Bazowych (Baselines)

Aby sprawdzić, czy uczenie głębokie faktycznie przynosi pożądane rezultaty, należy porównać wyniki sieci neuronowych z klasycznymi, analitycznymi algorytmami przetwarzania obrazów. Zaimplementowano następujące podejścia bazowe (tzw. baselines):

1. **Zwiększanie rozdzielczości (Super-Resolution Baseline):** Interpolacja bikubiczna (OpenCV `resize`). Metoda ta nie "wymyśla" nowych detali, a jedynie wygładza krawędzie rozciągniętego obrazu wejściowego za pomocą funkcji wielomianowych.
2. **Odszumianie (Denoising Baseline):** Filtracja bilateralna (`denoise_bilateral` z biblioteki `skimage` z parametrami `sigma_color=0.05` i `sigma_spatial=15`). Klasyczny algorytm usuwania szumów, który chroni ostre krawędzie, jednocześnie rozmywając jednorodne obszary.

Poniższy kod iteruje przez zbiór walidacyjny, w identyczny sposób obliczając nasze trzy metryki (PSNR, SSIM, LPIPS) dla metod tradycyjnych, i wypisuje je w formie tabeli. Dzięki temu możemy porównać je z tabelą otrzymaną z modeli głębokich.

In [3]:
from services import evaluate_baselines

df_baselines = evaluate_baselines()

print("\n--- Tabela wyników metod bazowych ---")
display(df_baselines)

Evaluation: OpenCV Bicubic Interpolation (Super-Resolution)...
Evaluation: skimage denoise_bilateral (Denoising)...

--- Tabela wyników metod bazowych ---


,Metoda,PSNR,SSIM,LPIPS
0,bicubic_interpolation_super_resolution,19.3285,0.4846,0.5288
1,denoise_bilateral_skimage,27.4192,0.9121,0.1506
